# Code vs Code - stage 1: can a browser-sized model build it?

Each candidate model plays the sample challenge ("Team notes") through its scripted
conversations, with exactly the prompt the game will give it, and the app it ends with is run
against the 40 hidden tests.

- **Part A** - every candidate at full precision under `transformers`. Quick to set up, and the
  ceiling: a 4-bit copy can only do as well or worse.
- **Part B** - the finalists converted by us to the 4-bit browser format (MLC, the format
  WebLLM loads) and measured again, on Colab's GPU. This is the number that counts, and it
  proves the conversion path that stage 8 needs for cvchelper.

Run on a **T4 GPU** (Runtime > Change runtime type). Part A takes roughly 10 to 20 minutes
per candidate with one seed. Everything is kept on the Colab machine under `/content/stage1/`,
and the last step packs it into zip files to download: the results (tables, every reply, every
app the models wrote) and the 4-bit model from Part B. Nothing is written to Google Drive.

A Colab session that disconnects loses its files, so download the zips before closing the tab.

No API key of any kind is used. Models are downloaded from public Hugging Face repos.

## Settings

In [ ]:
REPO = "https://github.com/MoAbboud/PortFolio.git"
BRANCH = "code-vs-code-updates"   # the branch with codevscode/ stage 1; "main" once merged

CANDIDATES = [
    "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "Qwen/Qwen3-1.7B",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
]
SEEDS = [0]            # 0 is greedy. Add 1, 2 for sampled runs, to see how much results move.
MAX_NEW_TOKENS = 4096  # a whole app.js is around 2,500 tokens

## Setup: GPU, the repo, Node and the challenge

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.isdir("PortFolio"):
    !git clone --depth 1 --branch {BRANCH} {REPO} PortFolio
else:
    !cd PortFolio && git pull --ff-only

In [ ]:
# Node 22 or newer: the hidden tests run in the same JavaScript runner as the game.
!node --version 2>/dev/null | grep -qE '^v(2[2-9]|[3-9][0-9])' || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null && apt-get install -y nodejs >/dev/null)
!node --version
!cd PortFolio/codevscode && npm ci --silent
!cd PortFolio/codevscode && node tools/validate.mjs challenges/sample

In [ ]:
import sys, json, time, datetime, pathlib
sys.path.insert(0, "/content/PortFolio/codevscode")
from codevscode.harness import load_challenge, measure

challenge = load_challenge("/content/PortFolio/codevscode/challenges/sample")
print(challenge.id, "-", list(challenge.scripts), "-", len(challenge.answers), "hidden requirements")

RUN = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
OUT = pathlib.Path(f"/content/stage1/{RUN}")   # on the Colab machine only
OUT.mkdir(parents=True, exist_ok=True)
print("saving to", OUT)

In [ ]:
def save(rows, part):
    """The table as JSON, and for every row the replies and the files the model ended with."""
    table = []
    for row in rows:
        played = row["replay"]
        folder = OUT / part / row["model"].replace("/", "__").replace(" ", "_") / row["script"]
        folder.mkdir(parents=True, exist_ok=True)
        for name, text in played.files.items():
            (folder / name).write_text(text, encoding="utf-8")
        (folder / "conversation.json").write_text(json.dumps(
            [{"message": t.message, "reply": t.reply, "wrote": t.wrote, "skipped": t.skipped,
              "seconds": round(t.seconds, 1)} for t in played.turns], indent=2), encoding="utf-8")
        table.append({k: v for k, v in row.items() if k != "replay"})
    (OUT / f"{part}.json").write_text(json.dumps(table, indent=2), encoding="utf-8")
    return table

COLUMNS = ["model", "script", "passed", "of", "loads", "correctness", "security", "efficiency", "practicality", "seconds"]

## Part A - every candidate at full precision

16-bit weights on the T4. Qwen3 runs with its thinking mode off: the helper should write code,
not a long chain of reasoning first.

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="cuda")
    model.eval()
    return tokenizer, model

def make_generate(name, tokenizer, model, seed):
    extra = {"enable_thinking": False} if "Qwen3" in name else {}
    def generate(messages):
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, **extra)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        torch.manual_seed(seed)
        sampling = {"do_sample": True, "temperature": 0.7, "top_p": 0.95} if seed else {"do_sample": False}
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **sampling)
        return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generate

def report(row):
    print(f"  {row['script']:16} {row['passed']:>2}/{row['of']}  loads={row['loads']}  "
          f"C {row['correctness']}  S {row['security']}  E {row['efficiency']}  P {row['practicality']}  "
          f"{row['seconds']} s")

rows_a = []
for name in CANDIDATES:
    print(f"\n=== {name}")
    started = time.time()
    tokenizer, model = load(name)
    for seed in SEEDS:
        for row in measure(challenge, make_generate(name, tokenizer, model, seed), f"{name} seed {seed}"):
            rows_a.append(row)
            report(row)
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    print(f"  ({time.time() - started:.0f} s)")
    save(rows_a, "part-a")

In [ ]:
import pandas as pd
table_a = pd.DataFrame(save(rows_a, "part-a"))
table_a[COLUMNS]

## Part B - the finalists, converted by us to the browser's 4-bit format

Pick the one or two best from Part A. Each is downloaded, converted with MLC LLM to `q4f16_1`
(the quantisation WebLLM's own builds use), and run on Colab's GPU through MLC - the same
runtime family as the browser - with the same harness.

The output folder (`dist/<model>-q4f16_1-MLC`) is what a browser downloads, so its size is the
download size. The last step zips it for you to download.

**This part is new and unverified.** MLC's wheel names and template names change between
releases. If a cell fails, send the error - the fix is usually a name.

In [ ]:
FINALISTS = ["Qwen/Qwen2.5-Coder-1.5B-Instruct"]   # set from Part A
QUANTISATION = "q4f16_1"
# MLC's conversation template per model family. Check against `mlc_llm gen_config --help`.
TEMPLATES = {"Qwen2.5": "qwen2", "Qwen3": "qwen2", "SmolLM2": "chatml"}

In [ ]:
import torch
cuda = torch.version.cuda.replace(".", "")[:3]   # for example "124"
print("CUDA", torch.version.cuda, "-> MLC wheels for cu" + cuda)
!pip install -q --pre -U -f https://mlc.ai/wheels mlc-llm-nightly-cu{cuda} mlc-ai-nightly-cu{cuda}
!python -c "import mlc_llm; print('mlc_llm imported')"

In [ ]:
def convert(name):
    short = name.split("/")[-1]
    template = next(t for family, t in TEMPLATES.items() if family in name)
    source, dist = f"models/{short}", f"dist/{short}-{QUANTISATION}-MLC"
    !huggingface-cli download {name} --local-dir {source} --quiet
    !mlc_llm convert_weight {source} --quantization {QUANTISATION} -o {dist}
    !mlc_llm gen_config {source} --quantization {QUANTISATION} --conv-template {template} -o {dist}
    size = sum(f.stat().st_size for f in pathlib.Path(dist).rglob("*") if f.is_file())
    print(f"{dist}: {size / 1e6:.0f} MB - the browser download")
    return dist, size

from mlc_llm import MLCEngine

rows_b = []
BUILDS = []   # the converted model folders, zipped by the last step
for name in FINALISTS:
    dist, size = convert(name)
    engine = MLCEngine(dist, device="cuda")   # compiles the CUDA library on first use
    for seed in SEEDS:
        def generate(messages, seed=seed):
            reply = engine.chat.completions.create(
                messages=messages, max_tokens=MAX_NEW_TOKENS,
                temperature=0.7 if seed else 0.0, top_p=0.95, seed=seed)
            return reply.choices[0].message.content
        for row in measure(challenge, generate, f"{name} {QUANTISATION} seed {seed}"):
            row["download_mb"] = round(size / 1e6)
            rows_b.append(row)
            report(row)
    engine.terminate()
    BUILDS.append(dist)
table_b = pd.DataFrame(save(rows_b, "part-b"))
table_b[COLUMNS + ["download_mb"]]

## Last step - the zips to download

Two files, so the small one arrives even if the big one is slow:

- `cvc-stage1-results-<run>.zip` - the tables, every conversation, every app the models wrote.
  A few MB.
- `cvc-stage1-model-<run>.zip` - the 4-bit model folder(s) from Part B, exactly what a browser
  loads. Several hundred MB for a 1.5B model. Only made if Part B ran.

Your browser asks where to save each one. If a download stalls, the same files are in the
file panel on the left under `/content/` - right-click, Download.

In [ ]:
import shutil
from google.colab import files

results_zip = shutil.make_archive(f"/content/cvc-stage1-results-{RUN}", "zip", OUT)
print(f"{results_zip}: {pathlib.Path(results_zip).stat().st_size / 1e6:.1f} MB")

model_zip = None
if globals().get("BUILDS"):
    staging = pathlib.Path(f"/content/cvc-model-{RUN}")
    for build in BUILDS:
        shutil.copytree(build, staging / pathlib.Path(build).name, dirs_exist_ok=True)
    model_zip = shutil.make_archive(f"/content/cvc-stage1-model-{RUN}", "zip", staging)
    print(f"{model_zip}: {pathlib.Path(model_zip).stat().st_size / 1e6:.0f} MB")
else:
    print("Part B has not run, so there is no model to zip yet.")

files.download(results_zip)
if model_zip:
    files.download(model_zip)